[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_GPU/Intro_GPU.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle**

# Intro to GPU Accelerated Scientific Computing

As the name implies, we will learn how to manage CPU (host) and GPU(device) interactions for performing GPU accelerated computing.

### Acknowledgements

First, I'd like to thank NVIDIA for creating CUDA, it's a super easy-to-learn API for using your GPU and accelerating computations. 

I'd like to thank UF and SPS for their interest in this student organization, giving everyone a platform to share their knowledge and resources.

Finally, I'd like to personally thank Dr. Silva, Dr. Principe, Benjamin Colburn, Dr. Wong, Dr. Shea, Matheus Kunzler-Maldaner and Evan Partidas for their continual support in my academic journey.

## 0. Introduction

Scientific Computing is increasingly important as topics such as data analysis, signal processing and machine learning gain relevance in research. 

Specifically, it's important for synthetic tests where evaluation data can be generated independently from other each-other while the computations are as parallelizable as possible.

Another case in which it's important is in any real-time system which is computation heavy and operates in a sequential but parallelizable manner. (We will discuss this more thoroughly soon).

The intent of this workshop is to introduce the user to basic libraries and APIs for GPU accelerating their own scripts, mainly by using the RAPIDS SDK for CuPy and the Numba API for GPU utilization.

## 1. Pre-requisites

There are actually very few pre-requisites for your understanding of using GPUs in your program. Potentially, having a Digital Design and Computer Architecture background will aid in your understanding of how to minimize latencies, however we will not discuss latencies in depth (if you are interested, be on the lookout for the *Hardware Accelerated Scientific Computing* and the *Introduction to Real Time Signal Processing* workshops TBA in the future!).

On the other hand, there are pre-requisites for following this workshop. At the very least, you should have a basic understanding of *Python* and if you don't, it still shouldn't be a major issue as it's almost pseudocode.

To get started, we will need to get your environment set up. We will be using the conda package manager to install the necessary packages. If you don't have conda installed, you can download it from here: https://www.anaconda.com/download/success

Now we need to actually setup the conda environment for the following workshop. Run the following commands (or their OS equivalent) in your terminal.

```
conda env create --name SPS
conda activate SPS
```

Then, we need to install the relevant libraries for this environment.

In [ ]:
%conda install ipykernel numba -c rapidsai -c nvidia -c conda-forge rapids=24.06 python=3.11 cuda-version=12.2

Make sure that your conda environment is selected as your jupyter kernel wherever that selection is made.

Now we are ready to get to work!

---
### 🕐 Session 1 of 2 — *From CPU to GPU* (~35 min)
**Goal:** understand SIMD, why GPUs exist, and how the host & device fit together (§2–§3.2).
**Builds on:** basic Python. &nbsp; **Feeds into:** Session 2 (memory, kernels & CuPy).

---

## 2. CPU

This is a section on CPU usage, specifically for computing. This section is relatively straightforward for the average programmer, but its purpose is to incentivize the usage of more scalable and efficient methods.

### 2.1. Regular

CPUs or *Central Processing Units* are what your computer uses for basically everything. They run all commands on your computer at a very fast pace (CPU clocks operate between 2 and 5 GHz, and can perform at least 1 instruction per clock cycle). 

Assuming the bare minimum, that is, the instructions being run are of SISD type (*Single Input Single Data*), your computer can perform at least 2 Billion instructions per second which is a pretty large amount of operations for such a short period of time. Let's generate an example and time it to put things into perspective.

In [ ]:

# YOUR CODE HERE


While some simple operations perform relatively well, it is clear that more complex computations (like Convolutions) can take over 10x as long.

Moreover, ponder the situation if you further need to scale these operations to a larger array size, or potentially even having multiple dimensions (as when we work with images or videos).

### 2.2. Vectorization

💡 **Intuition.** A regular CPU instruction is like a cashier scanning grocery items one at a time. Vectorized (SIMD) instructions are like sliding a whole tray of items over the scanner at once: *one* instruction, *many* data elements. NumPy is fast precisely because it hands your arrays to these tray-sized instructions instead of looping in Python.

To address the previous issues (from the CPU), CPU architectures were advanced to support vector-robust methods, or *Vectorized Instructions*. *Vectorized Instructions* are instructions with memory access patterns embedded into the instruction, that is that they are SIMD type (*Single Instruction Multiple Data*).

By instead making use of these kinds of instructions, we can make significant gains in performance - however, there are downsides to this. NumPy is built on top of C, which means that to take advantage of these improvements there need to be strict implications about the data's memory access. The primary downside is that Python Lists are made to handle objects and be mutable, whereas Vectorized-Access supported lists (which are associated with Numpy) must remain constant-sized and have a fixed data-type.

Let's see the results:

In [ ]:

# YOUR CODE HERE


We can see that there are some clear timing benefits of these implementations when directly compared, however by "playing" around with these numbers, these vectorized instructions are not very scalable and actually they fall short the larger the arrays are.

## 3. GPU

This is a section on GPU usage, specifically for computing. This section dives into the most fundamental topics involved in this kind of programming, and displays the scalability of these methods.

* **Note**: The code in this section involving GPU Device utility will often need to be ran twice. The reason for this is that the CPU needs to communicate to the GPU the *context* of the processing it will do. We will mention this again later.

### 3.1 Hardware Architecture

💡 **Intuition.** A CPU is a few *very fast, very clever* workers — great when tasks are sequential or unpredictable. A GPU is *thousands of simple workers* who all do the same thing at the same time — great when the same operation must hit millions of independent numbers (pixels, samples, weights). Neither is "better": the trick is matching the shape of your computation to the right worker pool.

First, some brief insights on the Hardware architecture to give some insights into how your computer system will work.

We mentioned that the CPU is the *Central Processing Unit* and this naming convention will be relevant as shown later. However, when it was introduced previously, some might be under the notion that the CPU is 1 unified piece of hardware, it is not.

Actually, modern CPUs have 2 main configurations:

*Single-Die, Multi-Core CPU* (SDMC CPU) or *Multi-Die, Multi-Core CPU* (MDMC CPU)

There are tradeoffs with each design, most notably: 

SDMC CPUs: **less** threads, **faster** clock speed

MDMC CPUs: **more** threads, **slower** clock speed

Within each die are multiple cores, which each contain multiple threads. Each *core* can run its own program, however certain kinds of programs can take advantage of the individual *threads* in a core to improve computational efficiency. We won't get into the specifics of threading yet, we will reserve that for a future workshop -

However, this should give some insight into the utility of a GPU. Originally named the *Graphics Processing Unit*, a GPU is really just a CPU optimized for vectorized operations.

In this case, if we were to continue the comparisons above:

MDMC CPUs: **more** threads, **slower** clock speed, **flexible** instruction set

GPUs: **even more** threads, **even slower** clock speed, **limited** instruction set

Hopefully, you have an idea about the structure for the system in your computer as a result of this.

### 3.2. System Structure

Recall that previously we called the CPU the *host* and the GPU the *device*, these are the terminology used to refer to these two components in our computers.

The CPU is considered the *host* because it queues instructions for the corresponding *device* to follow. The reason is simple, since the CPU operates at much higher clock speeds, it can queue instructions for the *device* to perform at a fast enough rate. 

The GPU is considered the *device* because it takes instructions from the *host* and distributes them among its multiple threads.

However, the structure of the GPU threads are further organized in comparison to CPUs. Typically, threads are organized into *warps* which are clusters of 32 threads that execute simultaneously - with this organization, it is possible to hide latencies due to data transfers or computations.

While the organization of threads into warps is meaningful in the context of hiding latencies, it is usually more useful to think about the organization of the threads in the context of your program.

---
### 🕐 Session 2 of 2 — *Memory, Kernels & CuPy* (~35 min)
**Goal:** move data host↔device deliberately; write a kernel with Numba; port NumPy code to CuPy and benchmark it (§3.3–§5).
**Builds on:** Session 1 (hardware picture). &nbsp; **Feeds into:** [Deep Learning for Physics](../Intro_DL_4_Physics/README.md).

---

### 3.3. GPU Memory

💡 **Intuition.** The GPU is fast, but it lives across town: every array you send it travels over the PCIe bus, and that trip usually costs more than the computation itself. The whole game of GPU programming is *minimizing trips* — send data once, do as much work as possible while it's there, and only bring back the final answer.

Now that we know the structure of the system, it should follow that there is a means to access data on the *device(s)* 

**Notes**: The largest latencies in any existing system are memory transfers. Therefore in general, we would like to avoid data transfers whenever possible. They are inevitable in **Real-Time Systems** where data is always handled *"globally"*, however the existence of *warps* help mitigate the latency introduced by these data transfers.

A very simple example:

In [ ]:

# YOUR CODE HERE


From this example, it may seem as though the GPU computations are not justified, they actually consume a lot of time in comparison to the CPU operations. Let's now compare a larger scale operation:

In [ ]:

# YOUR CODE HERE


**What just happened.** The headline number is enormous — NumPy convolution **263.1 s** against CuPy's **0.0035 s**, apparently a 75,000× speedup. **Both halves of that comparison are misleading, and unpicking them is the most useful thing in this workshop.**

**First problem: the timings around CuPy do not measure computation.** **CUDA kernel launches are asynchronous.** `cp.convolve(...)` queues work and returns *immediately*, before the GPU has done anything, so `time.perf_counter()` records **launch overhead** — a few microseconds — and not the convolution. The fix is one line: call `cp.cuda.Stream.null.synchronize()` before stopping the clock. **Notice that the Numba example in §3.5 does this correctly with `cuda.synchronize()`**, so the notebook contains both the right and the wrong pattern; compare them directly.

**Second problem: the two sides run *different algorithms*.** `np.convolve` is **direct** convolution, $O(NM)$ — with $N = 10^7$ and $M = 10^5$ that is $10^{12}$ multiply-adds, which is exactly why it takes 263 seconds. CuPy's convolve uses an **FFT-based** method, $O((N{+}M)\log(N{+}M))$ — about $10^9$ operations, a thousand-fold algorithmic advantage before any hardware is involved. **The comparison changes the algorithm and the processor at the same time.**

**So what is the honest number?** Run `scipy.signal.fftconvolve` on the CPU: same $O(N\log N)$ algorithm, same hardware class as the NumPy row, and it completes in roughly **one second**. Against a properly synchronised CuPy timing, the genuine GPU advantage on this problem is more like **10–100×** — still excellent, and an order of magnitude less than advertised. **A real speedup does not need an unfair baseline.**

**The rows that *are* trustworthy are the transfers, and they carry the session's actual lesson.** `cp.asarray(x)` on 10 million float64 values (80 MB) took 0.0138 s — about **5.8 GB/s**, consistent with PCIe. Compare that against the array operation at 0.0002 s. **Moving the data cost roughly 70× more than using it.** That is the whole reason the intuition cell says the game is minimising trips.

**Do the arithmetic that follows from it, because it decides whether GPU work pays.** At ~6 GB/s over the bus, a round trip for 80 MB costs about 27 ms. **Any computation that finishes in less than 27 ms is not worth sending** — you would spend more on postage than on the work. That single calculation, done before writing code, is what separates a useful port from a slower one.

**Note also why `cp.asarray` is fast here relative to the earlier small-array case.** The 0.147 s in §3.3 was dominated by **CUDA context initialisation on first use** — the notebook's own warning that GPU cells often need running twice. By this cell the context exists, so 0.0138 s is a genuine bandwidth measurement. **First-call timings on a GPU measure the driver, not your code**, which is why every serious benchmark warms up first.

**The transferable rule, in three parts.** Synchronise before you time. Compare like algorithm against like algorithm. And check correctness, not just speed — `np.allclose` against a CPU reference, which is precisely what §3.5 does and this cell does not.

Data transfer clearly introduces a large latency in the computation, but the reduction in complex computing is *massive*, over 300x more efficient (based on this example). Clearly, if the main latency is data transfers, then complex computations that are relevant to Machine Learning and Signal Processing can be significantly reduced by using GPUs. 

Before moving onto the most important part of using GPUs let's show how to transfer data back to the CPU.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three transfers back from device to host, and their times are wildly different:

| array | elements | bytes | time | rate |
|---|---|---|---|---|
| `x` | 10,000,000 | 80 MB | 0.0258 s | ~3.1 GB/s |
| `y` | 100,000 | 0.8 MB | 0.0003 s | ~2.7 GB/s |
| `z` | 10,099,999 | 81 MB | 0.0166 s | ~4.9 GB/s |

**These numbers are trustworthy in a way the CuPy compute timings are not, and it is worth saying why.** `cp.asnumpy` **must** wait for the data to actually arrive before returning a NumPy array, so it **synchronises implicitly**. The asynchronous-launch problem that invalidates the convolution timing above does not apply here — you cannot copy a result that has not been computed yet.

**Which makes these the workshop's honest measurement of the thing that matters most.** Three to five GB/s is ordinary PCIe territory, and the variation between rows is mostly measurement noise plus whether the source buffer was pageable or pinned. **Roughly 40 ms just to bring 160 MB home.**

**Put that beside the compute times and the design rule writes itself.** The array operation `2*x_d+1` on the same 10-million-element array took **0.0002 s**. The round trip to get that array there and back costs about **40 ms** — **200× the computation it enabled**. Sending data to a GPU to do one elementwise operation is strictly slower than staying on the CPU.

**So the arithmetic to do before porting anything is: how long would this take on the CPU, and how long is the transfer?** At ~5 GB/s, 80 MB costs 16 ms each way. **Any computation finishing in under ~30 ms is not worth sending.** That calculation takes ten seconds and prevents the most common disappointment in GPU programming — a careful port that runs slower than the original.

**And it explains the strategy the rest of the field follows.** Keep data **resident** on the device across many operations; batch small kernels into large ones; and only bring back the final answer. A deep-learning training loop moves one batch in and one scalar loss out while performing millions of operations in between — **that is a CGMA of thousands, and it is why GPUs transformed machine learning and not general scientific scripting.**

**One optimisation worth naming, since students will meet the flag.** Pinned (page-locked) host memory reaches roughly double this bandwidth and permits **asynchronous** transfers overlapped with computation — which is what `DataLoader(pin_memory=True)` requests in PyTorch. **The transfer never disappears; it can be hidden behind work**, and hiding it is the whole content of stream-based GPU programming.

### 3.4. GPU Kernels

💡 **Intuition.** A *kernel* is just the **body of a for-loop with the loop removed**. Instead of iterating `for i in range(N)`, you launch N threads and each thread asks "which `i` am I?" (that's the `threadIdx`/`blockIdx` arithmetic below) and then does the work for that one index. The `if thid < N` guard exists because threads come in fixed-size blocks, so a few extras may be launched past the end of your data.

Now that we have seen how to transfer data between the *host* and the *device*, we will dive into the main function of GPU Accelerated Computing.

In research, there are many times when you will need to implement your own new method. This new method, assuming that it is associated with **Signal Processing** or **Machine Learning**, will often operate on *vectors*, *matrices*, or *tensors* of all shapes and sizes, meaning that there can be a pattern-like memory access and possibility for Parallelization.

Here we will show an example of a kernel and express the key considerations when writing one.

In [ ]:

# YOUR CODE HERE


Above we built a fully functional kernel which generates a random FeatureMap, which is common in any machine learning or signal processing process.

Clearly, x and y are the 2 input vectors, and z is the 1 output tensor.

N, M and K are inputs that specify the shape of the corresponding input vector and the output tensor. It is possible to simply extract these values by using numba's built-in functions, but it is common practice the values in as an argument.

In a kernel, thidx, thidy and thidz are used to access input objects in a patterned way. It is not necessary to use all 3 indeces, you can use 2 or 1 depending on the program you are writing. 

Under extremely careful construction, it is likely that the if statement to check that the indeces are being correctly bounded are not necessary, however it is standard practice to code this restriction.

You may have noticed that each threadid is expressed as a sum of the threadIdx and some *block* information. The threads are the corresponding threads on the GPU which we have already discussed, however the blocks are clusters of threads, expressing how to "sparsify" the operations for the corresponding output. For very large matrices, it is clear that 1024 threads (the typical amount of threads in a GPU) will not suffice, which is why we need to *blockify* the operations for the output.

If you recall, *warps* are also clusters of threads. The main distinction between blocks and warps is that blocks simply express the total amount of resources that will be necessary to perform a kernel, whereas warps are the physical clusters of 32 threads to be ran immediately and independently from other warps.

Finally ```cuda.syncthreads()``` is a function which will force the GPU to wait for all the threads to finish computing before moving on. This is usually only necessary for more complex kernels with sequential operations.

### 3.5. GPU System

Now, we have discussed all the main tools necessary to create a system that uses a GPU. Let's see how we would use one in practice.

In [ ]:
#We are setting the seed to make the results reproducible for everyone
#We are creating random data, consider that in a real time system we would just have the data,
#but for synthetic data we would have to generate it here
#First let's set the data type and dimensions of the data
#Now we fill the data with random values
#Now, we need to allocate memory in the GPU
#Now we create the output array
#Now we transfer the data back to the host
#We can optionally print the results
#print(z)
#Instead, We'd like to emphasize that the computing time is low and accurate
#Let's compare the results
#print(f"Transfer latency: {(tic2-tic1)+(tic4-tic3):0.4f} seconds")

# YOUR CODE HERE


**What just happened.** A hand-written CUDA kernel produced $2^{16} \times 32 \times 16 = 33.5$ million values in **0.0311 s** against the CPU's **0.4961 s** — a **15.9× speedup** — and, crucially, `GPU == CPU: True`.

**Start with what this cell does right, because it is a model of how to benchmark GPU code.** It calls **`cuda.synchronize()`** before stopping the clock, so it measures actual computation rather than launch overhead — unlike the CuPy timings in §3.3. It includes the **transfers** inside the measured region, so the number is end-to-end rather than flattering. And it ends with **`np.allclose(z_gpu, z_cpu)`**, a correctness check. **Fast and wrong is the default outcome of GPU work**, and verifying against a CPU reference is what catches it.

**Now read the baseline before believing the speedup, because it is the weakest possible one.** The CPU side is a **Python double loop** over 2.1 million iterations, calling `math.exp` and `math.log` one element at a time. That is not "the CPU" — it is the Python interpreter. **A vectorised NumPy version is one line:**

```python
z_cpu = np.repeat((np.exp(x/20)[:, None] * np.log(y)[None, :])[:, :, None], K, axis=2)
```

which runs in a few tens of milliseconds — **comparable to, and possibly faster than, the GPU's 31 ms.** So the honest claim is **15.9× faster than Python**, not 15.9× faster than NumPy. Ask the room to predict which before showing them; the instinct is usually wrong.

**And the reason the GPU cannot win big here is worth deriving, because it is the session's central concept.** Count the arithmetic: one `exp`, one `log`, one multiply — about 3 FLOPs per output element. Count the memory: 4 bytes written per element. **The CGMA ratio is well under 1**, against the ~1000 a modern GPU needs to reach peak. **This kernel is memory-bandwidth-bound**, so it runs at a small fraction of the device's capability no matter how it is written.

**The transfer arithmetic confirms it.** The output is $33.5$M float32 values — **134 MB** — copied back at a few GB/s, which is roughly 30 ms all by itself. **Essentially the entire measured GPU time is the copy home.** The commented-out `Transfer latency` line reports 0.0310 s of the 0.0311 s total; the computation is effectively free and the postage is the whole bill.

**Which is precisely the lesson the transfer cells set up, now measured in a complete system.** A kernel with a low CGMA on a large output cannot beat a vectorised CPU implementation, because both are limited by moving bytes rather than by doing arithmetic. **The GPU wins when the same data is reused many times** — matrix multiplication, convolution, attention — and not when each byte is touched once.

**Two implementation details worth pointing at while the code is on screen.** The launch configuration `[(N//2, 1, 1), (2, M, K)]` gives $2 \times 32 \times 16 = 1024$ threads per block, the hardware maximum — a deliberate choice, not an arbitrary one. And `cuda.syncthreads()` inside the kernel synchronises threads **within a block**, not across the grid, and since it sits after the only write it **does nothing here**. Harmless, and the notebook's description of it as waiting for "all the threads" is too broad.

**Finally, the honest summary to leave the room with.** This cell demonstrates a correct, well-instrumented GPU workflow and a real 15.9× win over naive Python. **It does not demonstrate that GPUs beat NumPy on this problem** — and knowing which claim your benchmark supports is worth more than the number itself.

We just observed (depending on your machine) around or over 10x speedup in computing, and this is for a singular computation. This is clearly very good for scaling computing.

While the results are great and there are clear implications about this speedup, let's dive into the *kernel configuration*.

Typically function calls are made where you just call 

```function(...)``` 

However, kernels are called 

```kernel[(grid_dims),(block_dims)]```

Again returning to the concept of structure, *threads* are organized into *blocks*, and *blocks* are organized into a *grid*.

As mentioned previously, there are many occasions where a block does not iterate over the entire object, which is why a *grid* is necessary to express how to iterate through the object, in terms of the blocks.

Finally, the ```cuda.synchronize()``` function is necessary to ensure that kernels that depend on previous results do not begin until the necessary results have been generated.

## 4. Minimizing Latency

💡 **Intuition.** Once transfers dominate, speed comes from *choreography*, not raw compute: keep data resident on the device, batch small operations into big ones, and overlap transfers with computation so the GPU never sits idle waiting for the mail.

This section is mostly for completeness, but so we will not fully dive into this topic until the sequel workshop, *Hardware Accelerated Scientific Computing* (stay tuned). However we will mention key topics for consideration.

We've already seen an example of a *full system* that makes use of the GPU device, however we have only scratched the surface when it comes to optimizing the GPU's performance. 

It is possible that an implementation like in our example is the most optimal one, especially in cases where *a lot* of data is handled simultaneously. This mostly depends on your GPU's potential which you can check by running ```nvidia-smi``` in your terminal, however typically this is not the case.

There are ways to quantify the efficiency of your program, namely the *compute to global memory access ratio* (CGMA) which is defined as floating-point calculations : global memory access. 

Some intuition for this: High-End GPUs today have *memory-bandwidth* supporting between 500 and 1000 GB/s. On the other hand, they support between 600 and 1300 TFLOPS, that is tera *Floating-Point Operations per second*. This implies that **optimal** *CGMA* ratios should hang around 1000. 

There are a few ways to hide latency, one which was already addressed (refer to the warps details above), and the other is to build programs with strong consideration for memory transfers/access. A memory transfer from the CPU to the GPU is inevitable, however by making use of the GPU's local resources composed of *registers* and *shared-memory*, it is possible to further reduce memory latency.

We won't dive into access patterns yet, but let's define different memory types. *Device Global Memory* is the GPU's dedicated memory, it has the most amount of memory, implemented using DDR4, it is shared amongst all threads and it is the slowest of the 3. *Device Shared Memory* is the GPU's local memory which is shared amongst threads in a block - this means that as its name implies, the memory is shared amongst these threads, and it is faster than *Device Global Memory*. *Device Local Memory* is also the GPU's local memory which is private amongst the threads in a block, it can only be accessed by its own thread and is the fastest of the memory accesses.

In creating kernels, these are important considerations, however libraries like CuPy (in the next section), take care of most of this for us already.

## 5. NumPy (CPU) and CuPy (GPU)

While it can be a fun (and humbling) learning experience to implement kernels in Numba, we'd like in general to avoid making them from scratch because super corporations like NVIDIA have already implemented most Linear Algebra, Statistics or Machine Learning tools that you would ever need.

My intent is to just introduce some tools/functions that will accelerate your workflow.


```cp.asarray(x)``` sends a NumPy array to the device

```cp.asnumpy(x_d)``` sends a device array back to the host

```cp.empty((N,M,K), dtype=cp.float32)``` allocates a device array

```cp.zeros((N,M,K), dtype=cp.float32)``` instantiates a device array with 0s

```cp.repeat(X_d, K, axis=a)``` repeats a device array K times along axis a

```X_d@W_d``` performs a matrix multiplication

```X_d(*)W_d``` performs an element-wise operation (adding,subtracting, etc.)

CuPy and NumPy have very similar workflows, so actually most functions will work nearly identically as long as you specify to use NumPy instead:

```np.empty((N,M,K), dtype=np.float32)``` allocates a host array

```np.zeros((N,M,K), dtype=np.float32)``` instantiates a host array with 0s

```np.repeat(X, K, axis=a)``` repeats a host array K times along axis a

```X@W``` performs a matrix multiplication

```X(*)W``` performs an element-wise operation (adding,subtracting, etc.)

## 6. Conclusion

GPU computing is important because it enables the rapid processing of large-scale computations and data parallelism, significantly accelerating tasks like machine learning, scientific simulations, and image rendering. This enhances performance and efficiency in various fields, including AI, research, and gaming.

We covered topics including: memory management, kernel development and configuration, and workflow acceleration.

**Where next:**
- [Deep Learning for Physics](../Intro_DL_4_Physics/README.md) — train real models on the hardware you just learned to drive.
- [Foundations of Signal Processing](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — the FFT and convolution algorithms worth accelerating.
- [Intro to C](../Intro_Programming/Intro_C.ipynb) — the memory model underneath everything here.